# Exercise 1.2 – UAP Natural Language Processing

**Contents**

0. Setup
1. NLP pipeline on the Brazilian article (source details first)
2. Correcting NER errors with a rule-based list
3. Comparing the small and medium spaCy models
4. Context-based correction
5. Ambiguous names and generalisation test
6. Word Sense Disambiguation
7. Historical source: Project Blue Book

## About this notebook

The code is based on **Exercises 1.2 and 1.3** of the course below, recreated using UAP report texts instead of the course examples.

- **Course:** *Knowledge Graphs – Foundations and Applications*
- **Platform:** openHPI (Hasso Plattner Institute), self-paced since 21/11/2023
- **Instructors:** Prof. Dr. Harald Sack, Sasha Bruns, Mary Ann Tan, Tabea Tietz and Mahsa Vafaie (FIZ Karlsruhe – Leibniz Institute for Information Infrastructure & Karlsruhe Institute of Technology)
- **URL:** https://open.hpi.de/courses/knowledgegraphs2023
- **Date accessed:** 01/10/2026

Course material © the instructors, licensed under CC BY-NC 3.0 (Creative Commons Attribution-NonCommercial).

*All dates in this notebook are written as dd/mm/yyyy.*

## 0. Setup
Install the libraries and spaCy models (only needed once).

In [38]:
# Install spaCy and its English language model
!pip -q install spacy pandas
!python -m spacy download en_core_web_sm


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ----------- ---------------------------- 3.7/12.8 MB 27.4 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 40.1 MB/s  0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [39]:
%pip install spacy

Note: you may need to restart the kernel to use updated packages.


In [40]:
# Download the medium and large English models (only needed once)
!python -m spacy download en_core_web_md
!python -m spacy download en_core_web_lg

     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     --- ------------------------------------ 3.1/33.5 MB 26.4 MB/s eta 0:00:02
     ---------------- ---------------------- 14.4/33.5 MB 44.1 MB/s eta 0:00:01
     ------------------------------ -------- 26.2/33.5 MB 49.6 MB/s eta 0:00:01
     ---------------------------------------- 33.5/33.5 MB 48.7 MB/s  0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


     ---------------------------------------- 0.0/400.7 MB ? eta -:--:--
     ---------------------------------------- 1.0/400.7 MB 8.1 MB/s eta 0:00:50
     ---------------------------------------- 2.9/400.7 MB 8.4 MB/s eta 0:00:48
      -------------------------------------- 5.8/400.7 MB 10.3 MB/s eta 0:00:39
      -------------------------------------- 9.2/400.7 MB 12.2 MB/s eta 0:00:32
     - ------------------------------------ 14.2/400.7 MB 14.6 MB/s eta 0:00:27
     - ------------------------------------ 20.2/400.7 MB 17.3 MB/s eta 0:00:22
     -- ----------------------------------- 29.1/400.7 MB 21.0 MB/s eta 0:00:18
     --- ---------------------------------- 39.6/400.7 MB 25.0 MB/s eta 0:00:15
     ---- --------------------------------- 52.7/400.7 MB 29.4 MB/s eta 0:00:12
     ------ ------------------------------- 65.3/400.7 MB 32.6 MB/s eta 0:00:11
     ------- ------------------------------ 77.3/400.7 MB 34.9 MB/s eta 0:00:10
     -------- ----------------------------- 87.

In [41]:
%pip install geotext

Note: you may need to restart the kernel to use updated packages.


## Source: Brazilian "Official UFO Night"

- **Case:** Brazilian "Official UFO Night" (*Noite Oficial dos OVNIs*): radar and visual sightings
- **Date of event:** 19/05/1986

**Text analysed in Sections 1–4:**
- **Article:** "Official UFO Night in Brazil", gov.br (Government of Brazil, English-language portal), published 20/05/2022, no author named
- **URL:** https://www.gov.br/en/government-of-brazil/latest-news/2022/official-ufo-night-in-brazil
- **Date accessed:** 07/10/2026
- **Type:** Secondary source (government news article, written 36 years after the event)

**Other sources:**
- **Magazine:** Lyra, R. "Arquivos Abertos – OVNIs nos céus do Brasil". *Revista Imprensa Nacional – Novos Rumos da Comunicação Pública*, Year 1, No. 3, September 2017, pp. 33–34 (in Portuguese). Secondary source.
- **Primary source:** *Relatório de Ocorrências*, Comando Aéreo de Defesa Aérea (COMDA), Ministério da Aeronáutica, Brasília, 02/06/1986 (Official Letter No. 008/CMDO/C-138). Archive file BR_DFANBSB_ARX_0_0_0249_d0001de0001. Portuguese original; English working translation: machine translation by GPT-6.1, 11/10/2026 at 3:41 PM.

**Note:** The sources differ in some details (e.g. the gov.br article mentions four states, including Minas Gerais, and objects up to 100 metres; the magazine mentions three states). These differences will be represented as separate claims in the knowledge graph.

## 1. NLP pipeline on the Brazilian article
### 1.1 Load the model and the source text

In [1]:

import spacy
import pandas as pd
from spacy import displacy
from IPython.display import display

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Text from the Brazilian government article
text = (
    "On May 19, 1986, 21 unidentified flying objects, "
    "some of them up to 100 meters in diameter, "
    "were sighted by multiple witnesses, both civilian "
    "and military, in four states: São Paulo, "
    "Rio de Janeiro, Minas Gerais and Goiás."
)



### 1.2 Tokenisation

In [2]:
# Process the text
doc = nlp(text)

# Tokenisation
print("TOKENS:")
for token in doc:
    print(token.text)


TOKENS:
On
May
19
,
1986
,
21
unidentified
flying
objects
,
some
of
them
up
to
100
meters
in
diameter
,
were
sighted
by
multiple
witnesses
,
both
civilian
and
military
,
in
four
states
:
São
Paulo
,
Rio
de
Janeiro
,
Minas
Gerais
and
Goiás
.


### 1.3 Lemmatisation, POS tagging and dependencies

In [3]:
# Lemmatisation, POS tagging and dependency parsing
token_data = [
    {
        "Text": token.text,
        "Lemma": token.lemma_,
        "POS": token.pos_,
        "Dependency": token.dep_
    }
    for token in doc
]

display(pd.DataFrame(token_data))


,Text,Lemma,POS,Dependency
0,On,on,ADP,prep
1,May,May,PROPN,pobj
2,19,19,NUM,nummod
3,",",",",PUNCT,punct
4,1986,1986,NUM,nummod
5,",",",",PUNCT,punct
6,21,21,NUM,nummod
7,unidentified,unidentified,ADJ,amod
8,flying,fly,VERB,amod
9,objects,object,NOUN,appos


### 1.4 Named Entity Recognition

In [4]:
# Named Entity Recognition (NER)

entities = [
    {
        "Entity": ent.text,
        "Label": ent.label_,
        "Meaning": spacy.explain(ent.label_)
    }
    for ent in doc.ents
]

display(pd.DataFrame(entities))

,Entity,Label,Meaning
0,"May 19, 1986",DATE,Absolute or relative dates or periods
1,21,CARDINAL,Numerals that do not fall under another type
2,up to 100 meters,QUANTITY,"Measurements, as of weight or distance"
3,four,CARDINAL,Numerals that do not fall under another type
4,São Paulo,PERSON,"People, including fictional"
5,Minas Gerais,PERSON,"People, including fictional"
6,Goiás,GPE,"Countries, cities, states"


### 1.5 Visualising entities and dependencies

In [5]:
# Visualise named entities

print("NAMED ENTITIES:")
displacy.render(doc, style="ent", jupyter=True)


NAMED ENTITIES:


In [6]:
# Visualise dependency relations

print("DEPENDENCY RELATIONSHIPS:")
displacy.render(doc, style="dep", jupyter=True)

DEPENDENCY RELATIONSHIPS:


## 2. Correcting NER errors with a rule-based list
### 2.1 Inspect the original predictions

In [7]:
# Investigate the original NER predictions

for ent in doc.ents:
    print(
        f"Entity: {ent.text} | "
        f"Predicted: {ent.label_} | "
        f"Explanation: {spacy.explain(ent.label_)}"
    )


Entity: May 19, 1986 | Predicted: DATE | Explanation: Absolute or relative dates or periods
Entity: 21 | Predicted: CARDINAL | Explanation: Numerals that do not fall under another type
Entity: up to 100 meters | Predicted: QUANTITY | Explanation: Measurements, as of weight or distance
Entity: four | Predicted: CARDINAL | Explanation: Numerals that do not fall under another type
Entity: São Paulo | Predicted: PERSON | Explanation: People, including fictional
Entity: Minas Gerais | Predicted: PERSON | Explanation: People, including fictional
Entity: Goiás | Predicted: GPE | Explanation: Countries, cities, states


### 2.2 Add an EntityRuler with the four Brazilian states

In [8]:

# Create a separate pipeline to preserve
# the original model for comparison

nlp_corrected = spacy.load("en_core_web_sm")

ruler = nlp_corrected.add_pipe(
    "entity_ruler",
    after="ner",
    config={"overwrite_ents": True}
)

patterns = [
    {"label": "GPE", "pattern": "São Paulo"},
    {"label": "GPE", "pattern": "Rio de Janeiro"},
    {"label": "GPE", "pattern": "Minas Gerais"},
    {"label": "GPE", "pattern": "Goiás"}
]

ruler.add_patterns(patterns)




In [9]:
# Process the same original text
doc_corrected = nlp_corrected(text)

# Compare the original and corrected results
print("ORIGINAL PREDICTIONS:")
for ent in doc.ents:
    print(ent.text, ":", ent.label_)

print("\nCORRECTED PREDICTIONS:")
for ent in doc_corrected.ents:
    print(ent.text, ":", ent.label_)



ORIGINAL PREDICTIONS:
May 19, 1986 : DATE
21 : CARDINAL
up to 100 meters : QUANTITY
four : CARDINAL
São Paulo : PERSON
Minas Gerais : PERSON
Goiás : GPE

CORRECTED PREDICTIONS:
May 19, 1986 : DATE
21 : CARDINAL
up to 100 meters : QUANTITY
four : CARDINAL
São Paulo : GPE
Rio de Janeiro : GPE
Minas Gerais : GPE
Goiás : GPE


In [10]:
# Visualise corrected entities
displacy.render(
    doc_corrected,
    style="ent",
    jupyter=True
)

### 2.3 Results

Before the EntityRuler, the small model correctly labelled one of the four Brazilian states (Goiás); São Paulo and Minas Gerais were labelled as PERSON, and Rio de Janeiro was not found. After adding the list, all four states were labelled as geopolitical entities (GPE). The rule-based list worked because it labels the exact names it was given, without looking at the context.

However, this solution only works for Brazil, because the list contains only these four state names. For example, it would not label Belo Horizonte as GPE, because Belo Horizonte is not in the list: a rule-based list is only as good as the names I give it. It was also tested on one short sentence that does not contain ambiguity, so we are not sure whether it would behave the same in a larger dataset.

A context rule could help (see section 4); however, there could be ambiguities, and some prepositions have different functions in a sentence. Because of this, it could give us the wrong label. Also, it only corrects entities that were already found by the model; if the model misses Belo Horizonte, the correction would not work.

## 3. Comparing the small and medium spaCy models

In [11]:

import spacy
import pandas as pd
from IPython.display import display

# Load two different NLP models
nlp_small = spacy.load("en_core_web_sm")
nlp_medium = spacy.load("en_core_web_md")

# Use the same historical text
doc_small = nlp_small(text)
doc_medium = nlp_medium(text)

# Extract the predictions from each model
results = []

for model_name, doc_result in [
    ("Small", doc_small),
    ("Medium", doc_medium)
]:
    for ent in doc_result.ents:
        results.append({
            "Model": model_name,
            "Entity": ent.text,
            "Predicted Label": ent.label_
        })

display(pd.DataFrame(results))


,Model,Entity,Predicted Label
0,Small,"May 19, 1986",DATE
1,Small,21,CARDINAL
2,Small,up to 100 meters,QUANTITY
3,Small,four,CARDINAL
4,Small,São Paulo,PERSON
5,Small,Minas Gerais,PERSON
6,Small,Goiás,GPE
7,Medium,"May 19, 1986",DATE
8,Medium,21,CARDINAL
9,Medium,up to 100 meters,QUANTITY


In [12]:
# Compare NER: run the same text through spaCy's small and medium models

# Download the medium model (only needed once)
# !python -m spacy download en_core_web_md

# Load the medium model (md = medium-sized trained model used by spaCy)
nlp_md = spacy.load("en_core_web_md")

# Excerpt from the Brazilian article (own variable name, so it isn't overwritten)
brazil_text = (
    "On May 19, 1986, 21 unidentified flying objects, "
    "some of them up to 100 meters in diameter, "
    "were sighted by multiple witnesses, both civilian "
    "and military, in four states: São Paulo, "
    "Rio de Janeiro, Minas Gerais and Goiás."
)

# Process the text with both models
doc_sm = nlp(brazil_text)      # small model
doc_md = nlp_md(brazil_text)   # medium model

# Show the entities from each model as a table
print("SMALL MODEL (sm):")
display(pd.DataFrame(
    [{"Entity": ent.text, "Label": ent.label_} for ent in doc_sm.ents]
))

print("MEDIUM MODEL (md):")
display(pd.DataFrame(
    [{"Entity": ent.text, "Label": ent.label_} for ent in doc_md.ents]
))

SMALL MODEL (sm):


,Entity,Label
0,"May 19, 1986",DATE
1,21,CARDINAL
2,up to 100 meters,QUANTITY
3,four,CARDINAL
4,São Paulo,PERSON
5,Minas Gerais,PERSON
6,Goiás,GPE


MEDIUM MODEL (md):


,Entity,Label
0,"May 19, 1986",DATE
1,21,CARDINAL
2,up to 100 meters,QUANTITY
3,four,CARDINAL
4,São Paulo,PERSON
5,Rio de Janeiro,GPE
6,Minas Gerais,ORG
7,Goiás,PERSON


### What the results show

Both models correctly identified the date, numbers and measurements, but they struggled with the geopolitical entities: each model labelled only one of the four Brazilian states correctly, and it was a different state in each case (Goiás in the small model, Rio de Janeiro in the medium model).

The medium model fixed one error (it found Rio de Janeiro), but it made two new ones (Goiás as PERSON and Minas Gerais as ORG). A likely reason is that both models were trained on English web and news text, so they are not familiar with the names of Brazilian states. São Paulo may also look like a person's name, because Paulo is a common first name.

This means that, for the knowledge graph, we cannot trust either model's place labels without checking them. A larger model is not guaranteed to help, as the medium model shows. Possible fixes would be a rule-based list of Brazilian states (spaCy's EntityRuler), a Portuguese language model, or fine-tuning a model on labelled examples so that it learns to distinguish place names from people's names.

However, the rule-based solution only solves the problem for Brazil. A larger dataset with reports from multiple countries would need a global gazetteer and entity linking, which could raise another issue: deciding which place a name refers to.

### 3.1 Trying a gazetteer (GeoText)

In [13]:
from geotext import GeoText

text = (
    "On May 19, 1986, unidentified objects were "
    "reported in São Paulo, Rio de Janeiro, "
    "Minas Gerais and Goiás."
)

places = GeoText(text)

print ("Cities:", places.cities)
print ('Countries:', places.countries)

Cities: ['São Paulo', 'Rio de Janeiro', 'Goiás']
Countries: []


### 3.2 Results

GeoText found three cities (São Paulo, Rio de Janeiro and Goiás) and no countries. It found no countries because no country is named in the sentence; GeoText's country list only looks for sovereign nation names. It did not find Minas Gerais, probably because there is no city with that name in its list.

In the sentence, São Paulo, Rio de Janeiro and Goiás are states, not cities, but GeoText labelled them as cities because cities with the same names also exist. This shows that a gazetteer (a geographical dictionary used to look up places) can find known place names, but it cannot look at the surrounding text to determine which meaning is intended.

## 4. Context-based correction

In [14]:
# Context-based correction: relabel PERSON/ORG entities as GPE
# when the grammar shows they are used as places

PLACE_PREPS = {"in", "over", "near", "across", "above", "at", "from"}
PLACE_NOUNS = {"state", "city", "country", "town", "region", "province"}

def context_says_place(ent):
    tok = ent.root
    
# Climb through lists and appositions ("states: São Paulo, Rio de Janeiro, ...")
    
    while tok.dep_ in ("conj", "appos", "compound") and tok.head != tok:
        tok = tok.head
        if tok.lemma_ in PLACE_NOUNS:
            return True
            
# Object of a place preposition ("in Washington")
    
    return tok.dep_ == "pobj" and tok.head.lower_ in PLACE_PREPS

for ent in doc_md.ents:
    new_label = ent.label_
    if ent.label_ in ("PERSON", "ORG") and context_says_place(ent):
        new_label = "GPE (corrected by context)"
    print(f"{ent.text:<16} {ent.label_:<8} -> {new_label}")

May 19, 1986     DATE     -> DATE
21               CARDINAL -> CARDINAL
up to 100 meters QUANTITY -> QUANTITY
four             CARDINAL -> CARDINAL
São Paulo        PERSON   -> GPE (corrected by context)
Rio de Janeiro   GPE      -> GPE
Minas Gerais     ORG      -> GPE (corrected by context)
Goiás            PERSON   -> GPE (corrected by context)


### Results

I used context to give place names the right label. The rule looks at the words around each entity (sentence-level context): for example, a name that follows *in* or *over*, or that appears in a list after *states*, is probably a place. In this way, the rule can distinguish people's names from place names without a country-specific list, so it should also work for other countries.

Applied to the medium model's output, the rule corrected all three wrong labels, so all four Brazilian states were labelled as GPE. The other entities (May 19, 1986; 21; up to 100 meters; four; and Rio de Janeiro) were not changed.

**Limitations:**
- The rule can only correct entities that the model has found. For example, the `sm` model missed Rio de Janeiro entirely, so the rule could not fix it.
- The rule relies on the dependency parse. If the parser makes a mistake, the rule may also give the wrong label.
- Human language is ambiguous, and prepositions can have different meanings: in "the man in uniform", *in* does not introduce a place. For this reason, the rule only changes PERSON and ORG labels, not dates or numbers.
- The rule has only been tested on one sentence. It still needs to be tested on more examples (such as the Washington sentences and other reports) and with other spaCy models to make sure it works well in general.

## 5. Ambiguous names and generalisation test
### 5.1 Baseline: how the model labels "Washington"

In [15]:
import spacy

nlp = spacy.load("en_core_web_sm")

sentences = [
    "Washington reported seeing an unidentified object.",
    "The unidentified object was observed over Washington, D.C.",
    "A witness named Washington contacted the authorities.",
    "The aircraft departed from Washington."
]

for sentence in sentences:
    doc = nlp(sentence)

    print("\nSentence:", sentence)

    for ent in doc.ents:
        print("Entity:", ent.text, "| Label:", ent.label_)



Sentence: Washington reported seeing an unidentified object.
Entity: Washington | Label: GPE

Sentence: The unidentified object was observed over Washington, D.C.
Entity: Washington | Label: GPE
Entity: D.C. | Label: GPE

Sentence: A witness named Washington contacted the authorities.
Entity: Washington | Label: GPE

Sentence: The aircraft departed from Washington.
Entity: Washington | Label: GPE


In [16]:
sentences = [
    "Washington reported the incident.",
    "Mr. Washington reported the incident.",
    "Pilot James Washington reported the incident.",
    "The incident occurred in Washington.",
    "The incident occurred in Washington, D.C."
]
for sentence in sentences:
    doc_washington = nlp_small(sentence)   # own name, so the Brazilian "doc" is not overwritten

    print('\n', sentence)

    for ent in doc_washington.ents:
        print(ent.text, '->', ent.label_)


 Washington reported the incident.
Washington -> GPE

 Mr. Washington reported the incident.
Washington -> PERSON

 Pilot James Washington reported the incident.
James Washington -> PERSON

 The incident occurred in Washington.
Washington -> GPE

 The incident occurred in Washington, D.C.
Washington -> GPE
D.C. -> GPE


### 5.2 Applying the context rule to the Washington sentences

In [17]:
# Test whether the context rule generalises to new sentences

test_sentences = [
    "Washington reported the incident.",
    "Mr. Washington reported the incident.",
    "Pilot James Washington reported the incident.",
    "The incident occurred in Washington.",
    "The incident occurred in Washington, D.C.",
]

for sentence in test_sentences:
    doc_test = nlp_md(sentence)
    print("\n", sentence)
    for ent in doc_test.ents:
        new_label = ent.label_
        if ent.label_ in ("PERSON", "ORG") and context_says_place(ent):
            new_label = "GPE (corrected by context)"
        print(f"   {ent.text:<22} {ent.label_:<8} -> {new_label}")



 Washington reported the incident.
   Washington             GPE      -> GPE

 Mr. Washington reported the incident.
   Washington             PERSON   -> PERSON

 Pilot James Washington reported the incident.
   James Washington       PERSON   -> PERSON

 The incident occurred in Washington.
   Washington             GPE      -> GPE

 The incident occurred in Washington, D.C.
   Washington             GPE      -> GPE
   D.C.                   GPE      -> GPE


### Results

On the Washington sentences, the model already labelled every entity correctly, so the rule did not need to change anything. This shows that the rule is safe: it did not turn people's names into places. However, it does not yet show that the rule can fix errors in new sentences. The model was trained on English text, so this test does not tell us whether the rule would help with place names from other languages.

I noticed that sentence 1 is ambiguous: "Washington" can refer to a place, a person or the US government, and without more context it is difficult to know which.

It would be interesting to test the approach on reports written in other languages, such as Portuguese, French, German and Chinese. This would require a language-specific model and translated context words for each language. It would also be useful to run the model and the rule on English sentences containing place names from other countries, such as China, France and Germany.

## 6. Word Sense Disambiguation
### 6.1 "light": WordNet senses and the Lesk algorithm

In [18]:

import nltk
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk

nltk.download("wordnet")

# Use specific names ("light_...") so the "track" cells in 6.2 do not overwrite these sentences

light_sentence_a = (
    "The pilot observed a bright light "
    "moving across the sky."
)

light_sentence_b = (
    "The investigator used a light "
    "to examine the aircraft."
)


[nltk_data] Downloading package wordnet to
[nltk_data]   Package wordnet is already up-to-date!


In [19]:
# Find possible meanings of "light"
print("WORDNET DEFINITIONS:")

for sense in wn.synsets("light", pos=wn.NOUN):
    print(sense.name(), ":", sense.definition())

WORDNET DEFINITIONS:
light.n.01 : (physics) electromagnetic radiation that can produce a visual sensation
light.n.02 : any device serving as a source of illumination
light.n.03 : a particular perspective or aspect of a situation
luminosity.n.01 : the quality of being luminous; emitting or reflecting light
light.n.05 : an illuminated area
light.n.06 : a condition of spiritual awareness; divine illumination
light.n.07 : the visual effect of illumination on objects or scenes as created in pictures
light.n.08 : a person regarded very fondly
light.n.09 : having abundant light or illumination
light.n.10 : mental understanding as an enlightening experience
sparkle.n.01 : merriment expressed by a brightness or gleam or animation of countenance
light.n.12 : public awareness
inner_light.n.01 : a divine presence believed by Quakers to enlighten and guide the soul
light.n.14 : a visual warning signal
lighter.n.02 : a device for lighting or igniting fuel or charges or fires


In [20]:
# Apply the Lesk algorithm
light_sense_a = lesk(light_sentence_a.split(), "light", pos="n")
light_sense_b = lesk(light_sentence_b.split(), "light", pos="n")

print("\nSENTENCE A:")
print(light_sense_a.definition() if light_sense_a else "No sense found")

print("\nSENTENCE B:")
print(light_sense_b.definition() if light_sense_b else "No sense found")
print("\nSENTENCE A:", light_sense_a.name() if light_sense_a else "None")
print("SENTENCE B:", light_sense_b.name() if light_sense_b else "None")



SENTENCE A:
the quality of being luminous; emitting or reflecting light

SENTENCE B:
a divine presence believed by Quakers to enlighten and guide the soul

SENTENCE A: luminosity.n.01
SENTENCE B: inner_light.n.01


### Results: "light"

For sentence A ("The pilot observed a bright light moving across the sky."), Lesk chose luminosity.n.01, which means "the quality of being luminous; emitting or reflecting light". This is incorrect because the pilot sees a light in the sky, but "the quality of being luminous" describes a quality, not something that can move across the sky.

For sentence B ("The investigator used a light to examine the aircraft."), Lesk chose inner_light.n.01, which means "a divine presence believed by Quakers to enlighten and guide the soul". This is wrong because the investigator is using something that produces light, for example, a torch, to investigate the aircraft.

Compared with *track*, where Lesk chose the correct sense in 1 of 2 sentences, Lesk performed worse on *light* (0 of 2 correct). A possible reason is ambiguity: *light* has more senses in WordNet than *track*, so it is more difficult for Lesk to choose the right one.

### 6.2 "track": WordNet senses and the Lesk algorithm

In [21]:

import nltk
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk

nltk.download("wordnet")

sentence_a = "The radar tracked an unidentified object."
sentence_b = "The investigator tracked the development of the case."

# Explore possible meanings of the verb "track"
print("POSSIBLE MEANINGS:")

for sense in wn.synsets("track", pos=wn.VERB):
    print(sense.name(), ":", sense.definition())

# Apply Word Sense Disambiguation
sense_a = lesk(sentence_a.split(), "track", pos="v")
sense_b = lesk(sentence_b.split(), "track", pos="v")

print("\nSENTENCE A:")
print(sense_a.definition() if sense_a else "No sense found")

print("\nSENTENCE B:")
print(sense_b.definition() if sense_b else "No sense found")


POSSIBLE MEANINGS:
track.v.01 : carry on the feet and deposit
track.v.02 : observe or plot the moving path of something
chase.v.01 : go after with the intent to catch
traverse.v.01 : travel across or pass over
track.v.05 : make tracks upon

SENTENCE A:
carry on the feet and deposit

SENTENCE B:
observe or plot the moving path of something


[nltk_data] Downloading package wordnet to
[nltk_data]   Package wordnet is already up-to-date!


In [22]:

import nltk
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk

nltk.download("wordnet")

sentence_a = "The radar tracked an unidentified object."
sentence_b = "The investigator tracked the development of the case."

for name, sentence in [
    ("Radar observation", sentence_a),
    ("Case investigation", sentence_b)
]:
    sense = lesk(
        sentence.split(),
        "track",
        pos=wn.VERB
    )

    print(f"\n{name}")
    print("Sentence:", sentence)
    print("WordNet sense:", sense.name() if sense else "None")
    print("Definition:", sense.definition() if sense else "None")



Radar observation
Sentence: The radar tracked an unidentified object.
WordNet sense: track.v.01
Definition: carry on the feet and deposit

Case investigation
Sentence: The investigator tracked the development of the case.
WordNet sense: track.v.02
Definition: observe or plot the moving path of something


[nltk_data] Downloading package wordnet to
[nltk_data]   Package wordnet is already up-to-date!


### 6.3 Inspecting word overlap

In [23]:
# Import the libraries.

import re
import nltk
from nltk.corpus import wordnet as wn

#Downloads the WordNet data if it is not already installed. 

nltk.download("wordnet")

#Create our dataset

sentences = [
    "The radar tracked an unidentified object.",
    "The investigator tracked the development of the case."
]

#Define a function

def inspect_overlap(sentence):
    
#Extract the words
    
    context = set(
        re.findall(r"\b\w+\b", sentence.lower())
    )

    print("\nSENTENCE:", sentence)

    for sense in wn.synsets("track", pos=wn.VERB):
        
#Extract the words from each definition
        
        definition = set(
            re.findall(r"\b\w+\b", sense.definition().lower())
        )
        
#Find overlapping words
        
        overlap = context.intersection(definition)

#Display the results
        
        print("\nSense:", sense.name())
        print("Definition:", sense.definition())
        print("Overlapping words:", sorted(overlap))
        print("Overlap count:", len(overlap))

for sentence in sentences:
    inspect_overlap(sentence)



SENTENCE: The radar tracked an unidentified object.

Sense: track.v.01
Definition: carry on the feet and deposit
Overlapping words: ['the']
Overlap count: 1

Sense: track.v.02
Definition: observe or plot the moving path of something
Overlapping words: ['the']
Overlap count: 1

Sense: chase.v.01
Definition: go after with the intent to catch
Overlapping words: ['the']
Overlap count: 1

Sense: traverse.v.01
Definition: travel across or pass over
Overlapping words: []
Overlap count: 0

Sense: track.v.05
Definition: make tracks upon
Overlapping words: []
Overlap count: 0

SENTENCE: The investigator tracked the development of the case.

Sense: track.v.01
Definition: carry on the feet and deposit
Overlapping words: ['the']
Overlap count: 1

Sense: track.v.02
Definition: observe or plot the moving path of something
Overlapping words: ['of', 'the']
Overlap count: 2

Sense: chase.v.01
Definition: go after with the intent to catch
Overlapping words: ['the']
Overlap count: 1

Sense: traverse.v.01

[nltk_data] Downloading package wordnet to
[nltk_data]   Package wordnet is already up-to-date!


### 6.4 Improved Lesk

In [24]:
# Improved Lesk: stopwords removed, lemmatisation, definitions + examples

import re
import nltk
from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# Clean a text: lowercase, remove punctuation and stopwords, lemmatise

def clean_words(text):
    words = re.findall(r"\b\w+\b", text.lower())
    words = [w for w in words if w not in stop_words]
    
# lemmatise as a noun first, then as a verb (objects -> object, tracked -> track)
    
    return {lemmatizer.lemmatize(lemmatizer.lemmatize(w, "n"), "v") for w in words}

# Choose the sense with the largest meaningful overlap

def improved_lesk(sentence, word, pos=wn.VERB):
    context = clean_words(sentence)
    context.discard(word)   # the target word itself should not count as evidence

    best_sense, best_overlap = None, set()
    print("\nSENTENCE:", sentence)

    for sense in wn.synsets(word, pos=pos):
        signature = clean_words(sense.definition() + " " + " ".join(sense.examples()))
        overlap = context & signature
        print(f"  {sense.name():<15} overlap: {sorted(overlap)}")

        if len(overlap) > len(best_overlap):
            best_sense, best_overlap = sense, overlap

    return best_sense, best_overlap

# Run it on both sentences

for sentence in sentences:
    sense, overlap = improved_lesk(sentence, "track")
    if sense:
        print("  -> PREDICTION:", sense.name(), "|", sense.definition())
    else:
        print("  -> No overlap with any sense (Lesk cannot decide)")


SENTENCE: The radar tracked an unidentified object.
  track.v.01      overlap: []
  track.v.02      overlap: []
  chase.v.01      overlap: []
  traverse.v.01   overlap: []
  track.v.05      overlap: []
  -> No overlap with any sense (Lesk cannot decide)

SENTENCE: The investigator tracked the development of the case.
  track.v.01      overlap: []
  track.v.02      overlap: []
  chase.v.01      overlap: []
  traverse.v.01   overlap: []
  track.v.05      overlap: []
  -> No overlap with any sense (Lesk cannot decide)


[nltk_data] Downloading package stopwords to
[nltk_data]   Package stopwords is already up-to-date!


In [25]:
# Show every WordNet meaning of the verb "track"
for sense in wn.synsets("track", pos=wn.VERB):
    print(sense.name(), ":", sense.definition())
    print("   other words with this meaning:", sense.lemma_names())

track.v.01 : carry on the feet and deposit
   other words with this meaning: ['track']
track.v.02 : observe or plot the moving path of something
   other words with this meaning: ['track']
chase.v.01 : go after with the intent to catch
   other words with this meaning: ['chase', 'chase_after', 'trail', 'tail', 'tag', 'give_chase', 'dog', 'go_after', 'track']
traverse.v.01 : travel across or pass over
   other words with this meaning: ['traverse', 'track', 'cover', 'cross', 'pass_over', 'get_over', 'get_across', 'cut_through', 'cut_across']
track.v.05 : make tracks upon
   other words with this meaning: ['track']


### Results: how the meaning of a word changes with context

**Observations:**
- The word *light* can have different meanings depending on how it is used.
- The word *tracked* can mean something different when describing radar observations and when describing the development of an investigation.

**Results:**

| Sentence | Correct sense (ground truth) | NLTK Lesk | Correct? | Improved Lesk |
|---|---|---|---|---|
| The radar tracked an unidentified object. | track.v.02 – observe or plot the moving path of something | track.v.01 – carry on the feet and deposit | No | Cannot decide |
| The investigator tracked the development of the case. | track.v.02 – observe or plot the moving path of something (figurative) | track.v.02 – observe or plot the moving path of something | Yes, but probably by chance | Cannot decide |

**Interpretation:**
The algorithm could not find enough shared words to decide which meaning of *track* was intended. In the first sentence, "The radar tracked an unidentified object.", the algorithm compared the sentence with five possible definitions of *track*, for example track.v.02, "observe or plot the moving path of something". After removing common words such as *the*, *an* and *of*, none of the five definitions shared any of the remaining words in the sentence. This happened for both sentences, because there was not enough information to choose from.

NLTK Lesk was wrong for sentence A and correct for sentence B, but its correct answer was not based on meaningful evidence, because no important words overlapped with any definition. The difference between the two versions is that the improved version reports that no definition matched, while NLTK Lesk always returns a definition, even when the evidence is weak.

For sentence B, no WordNet sense describes monitoring progress directly. I chose track.v.02 because it is the closest meaning among the WordNet senses: following how a case develops can be understood figuratively as observing its "moving path".

In conclusion, matching words is not the same as interpreting meaning.

## 7. Historical source: Project Blue Book

- **Case:** Washington, D.C. radar and visual sightings
- **Date of event:** night of 26/07/1952 to 27/07/1952
- **Document type:** USAF incident report (transcription of original Project Blue Book document)
- **Source / archive:** Project 1947 website (transcription); originals held by the US National Archives
- **URL:** https://www.project1947.com/fig/1952c.htm
- **Date accessed:** 24/10/2026

In [26]:
# 7.1 Blue Book text: incident report, Washington D.C., night of 26/27 July 1952
# Source: Project 1947 transcription (see provenance cell above)
# Triple quotes are used because the text contains quotation marks

bluebook_text = """1. Varying numbers (up to 12 simultaneously) of u/i targets on ARTC radar scope. Termed by CAA personnel as "generally, solid returns", similar to a/c return except slower. No definable pattern of maneuver except at very beginning about 2150 EDT, 4 targets in rough line abreast with about 1 1/2 mile spacing moved slowly together (giving about 1" trace persistency at an estimated speed of less than 100 mph) on a heading of 110. At the same time 8 other targets were scattered throughout scope. ARTC checked Andrews Approach Control by telephone at 2200 EDT, following which weak and sporadic (unsteady) returns were picked up intermittently for another 3+ hours. Washington National Tower radar crew reports only one target positively u/i. This return was termed a "very good target" which moved across the scope from West to East at about 30 to 40 mph. However, the radar operators stated that there could have been other u/i targets on their scopes, particularly outside their area of a/c control, which they would not have noticed or would have assumed to be a/c under ARTC Center control. However, they noticed no other unusual (i.e. very slow or erratic) returns. ARTC Center controllers also report that a CAA flight inspector, Mr Bill Schreve, flying a/c #NC-12 reported at 2246 EDT that he had visually spotted 5 objects giving off a light glow ranging from orange to white; his altitude at time was 2200'. Some commercial pilots reported visuals ranging from "cigarette glow" (red-yellow) to "a light" (as recorded from their conversations with ARTC controllers). At 2238 EDT the USAF Command Post was notified of ARTC targets. Command Post notified ADC and KADF at 2245, and 2 F-94s were scrambled from Newcastle at 2300 EDT. ARTC controlled F-94's after arrival in area and vectored them to targets with generally negative results (flew through "a batch of radar returns" without spotting anything). However, one pilot mentioned seeing 4 lights at one time and a second time as seeing a single light ahead but unable to close whereupon light "went out" (these comments from ARTC controllers). One ARTC controller worked a USAF B-25 (AF8898 ?) for about 1 hour 20 mins about 2230 EDT. B-25 was vectored in on numerous targets and commented that each vector took him over a busy highway or intersection. Maj Fournet (AFOIN-2A2) and Lt. Holcomb (USN, AFOIN-2C5) arrived at ARTC Center about 27/0015 EDT. Lt. Holcomb observed scopes and reported "7 good, solid targets". He made a quick check with airport Weather Station and determined that there was a slight temperature inversion (about 1°) from the surface to about 1000'. However, he felt that the scope targets at that time were not the result of this inversion and so advised the Command Post with the suggestion that a second intercept flight be requested. (2nd intercept flight controlled by ARTC, but no strong targets remained when they arrived. They were vectored on dim targets with negative results.) Maj. Fournet and Lt. Holcomb remained in ARTC Center until 0415, but no strong targets were picked up; many dim and unstable targets (assumed due to temperature inversion) were observed throughout the remainder of the period."""

# Quick check: the text loaded correctly
print(len(bluebook_text.split()), "words")
print(bluebook_text[:100], "...")

530 words
1. Varying numbers (up to 12 simultaneously) of u/i targets on ARTC radar scope. Termed by CAA perso ...


### 7.2 NER predictions (before running the model)

| Entity | Expected label (what it is) | My prediction (what the model will say) | Why |
|---|---|---|---|
| Washington National Tower | FAC | GPE (only "Washington") | "Washington" is very common in news text as a city, so the model may ignore "National Tower". |
| Andrews Approach Control | FAC | PERSON | "Andrews" is also a surname, so the model may label it PERSON and ignore "Approach Control". |
| Mr Bill Schreve | PERSON | PERSON | "Mr" + first name + surname is a common pattern in news text, and Bill is a common American first name; the model was trained on English text. |
| Maj Fournet | PERSON | ORG | "Maj" is an abbreviation without a full stop, and Fournet is an unusual surname. I believe the model will confuse it with an organisation because of the abbreviation "Maj". |
| Newcastle | GPE | GPE | It looks like a known place name, but the model cannot tell which Newcastle it is (here, New Castle in Delaware, not Newcastle in England). |
| CAA | ORG | ORG | CAA is a well-known acronym, but today it usually refers to the UK Civil Aviation Authority, not the 1952 US Civil Aeronautics Administration. The model may label it correctly as ORG, but for the wrong organisation. |
| 2150 EDT | TIME | TIME | Four-digit times appear often in military and aviation texts, and "EDT" (a time zone) signals that it is a time. |
| 2200' | QUANTITY | TIME | It is an altitude (2,200 feet), but it looks like the four-digit times nearby (2150, 2200 EDT), so the model may label it TIME. |

In [27]:
# 7.3 Run NER on the Blue Book text (small and medium models)

doc_bb_sm = nlp(bluebook_text)      # small model
doc_bb_md = nlp_md(bluebook_text)   # medium model

# The 8 entities I made predictions for
my_entities = ["Washington National Tower", "Andrews Approach Control", "Bill Schreve",
               "Fournet", "Newcastle", "CAA", "2150 EDT", "2200'"]

def find_label(doc, target):
    # Find where the target appears in the text (character positions)
    start = doc.text.find(target)
    if start == -1:
        return "target not found in text"
    end = start + len(target)
    # Keep only entities that overlap that exact position
    found = [f"{ent.text} ({ent.label_})" for ent in doc.ents
             if ent.start_char < end and ent.end_char > start]
    return ", ".join(found) if found else "— (missed)"

# Show what each model said for each of my 8 entities
display(pd.DataFrame([
    {"Entity": t, "sm": find_label(doc_bb_sm, t), "md": find_label(doc_bb_md, t)}
    for t in my_entities
]))

,Entity,sm,md
0,Washington National Tower,Washington National Tower (ORG),Washington National Tower (FAC)
1,Andrews Approach Control,— (missed),Andrews Approach Control (ORG)
2,Bill Schreve,Bill Schreve (PERSON),Bill Schreve (PERSON)
3,Fournet,— (missed),Maj Fournet (PERSON)
4,Newcastle,Newcastle (GPE),Newcastle (GPE)
5,CAA,CAA (ORG),— (missed)
6,2150 EDT,about 2150 EDT (TIME),about 2150 EDT (CARDINAL)
7,2200',2200 (DATE),— (missed)


In [28]:
# 7.3 (continued) Large model: compare the small, medium and large models
nlp_lg = spacy.load("en_core_web_lg")
doc_bb_lg = nlp_lg(bluebook_text)

display(pd.DataFrame([
    {"Entity": t,
     "sm": find_label(doc_bb_sm, t),
     "md": find_label(doc_bb_md, t),
     "lg": find_label(doc_bb_lg, t)}
    for t in my_entities
]))

,Entity,sm,md,lg
0,Washington National Tower,Washington National Tower (ORG),Washington National Tower (FAC),Washington National Tower (ORG)
1,Andrews Approach Control,— (missed),Andrews Approach Control (ORG),Andrews Approach Control (ORG)
2,Bill Schreve,Bill Schreve (PERSON),Bill Schreve (PERSON),Bill Schreve (PERSON)
3,Fournet,— (missed),Maj Fournet (PERSON),— (missed)
4,Newcastle,Newcastle (GPE),Newcastle (GPE),Newcastle (GPE)
5,CAA,CAA (ORG),— (missed),CAA (ORG)
6,2150 EDT,about 2150 EDT (TIME),about 2150 EDT (CARDINAL),about 2150 EDT (TIME)
7,2200',2200 (DATE),— (missed),2200 (CARDINAL)


### 7.4 Results and comparison with the Brazilian text

All three models scored 4 out of 8, but they made different mistakes. The medium model was the only one that recognised Washington National Tower as FAC and Maj Fournet as PERSON. No model recognised 2200' (altitude) as QUANTITY, probably because the symbol ' (meaning feet) is rarely seen in the modern texts the models were trained on. A larger model did not perform better: the lg model has more word vectors, but it still performed the same as sm. Compared with the Brazilian text, where each model labelled only one of the four states correctly, the models performed better on the Blue Book text. This suggests that the models can find English names better than Portuguese ones.

### 7.5 Limitations

- Only one paragraph from the Blue Book file was analysed, so I cannot say that the models always behave like this on historical reports. I would need a larger dataset with more reports to compare how the models behave.
- The text is a transcription (made by the Project 1947 website), not the original scanned document, so it may contain copying errors or differences from the original that I cannot check without the scan.
- The transformer model (en_core_web_trf) could not be loaded in this environment because of a compatibility problem between Python 3.14 and PyTorch, so only the sm, md and lg models were compared.
- The expected labels were decided by me, so another person could decide differently and obtain different results. The labels would be more reliable if a second person labelled them independently.